# 3. Attribution (AU) and the 10-text rule

In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from polaudit import run_dataset
from polaudit.toy import toy_ratings, toy_spec

pd.set_option("display.width", 160)
result = run_dataset(toy_spec(), raw=toy_ratings())        # the toy dataset: 60 texts, 24 annotators, gender / politics / age

In [2]:
result.per_dimension

,texts,survived (AU),not survived,groups kept,groups removed
gender,60,20,40,2,-
politics,60,20,40,2,-
age,60,0,60,2,-


Politics and gender each attribute the 20 texts that were built to be polarized by them. Age attributes none: no text is split by age, so both age groups stay with 0 texts (every group is below 10, so the rule cannot choose one to remove).

In [3]:
result.per_group

AU texts  toxic  civil
dimension group                        
gender    man          20     10     10
          woman        20     10     10
politics  left         20     10     10
          right        20     10     10
age       old           0      0      0
          young         0      0      0

## The rounds of the 10-text rule

Raise the minimum so that the rule has something to do: with `min_au_texts=25` no group of the toy data reaches it, and since every group is below it, nothing is removed.

In [4]:
strict = run_dataset(toy_spec(min_au_texts=25), raw=toy_ratings())
strict.pruned["politics"].history

,round,group removed before this round,AU texts,groups,AU texts per group
0,0,None,20,2,"{'left': 20, 'right': 20}"


## Change the rules

Every rule is an argument. `min_au_texts=0` keeps every group (all AU texts, no 10-text rule), `min_au_texts=5` lowers the minimum, and `min_ratings` or `min_groups` make the test stricter. See `docs/parameters.md`.

In [5]:
for options in ({}, {"min_au_texts": 0}, {"min_au_texts": 5}, {"min_ratings": 13}):
    r = run_dataset(toy_spec(), raw=toy_ratings(), **options)
    print(options or "default", "->", len(r.au), "attributed rows, groups kept:", r.per_dimension["groups kept"].to_dict())

default -> 40 attributed rows, groups kept: {'gender': 2, 'politics': 2, 'age': 2}
{'min_au_texts': 0} -> 40 attributed rows, groups kept: {'gender': 2, 'politics': 2, 'age': 2}


{'min_au_texts': 5} -> 40 attributed rows, groups kept: {'gender': 2, 'politics': 2, 'age': 2}
{'min_ratings': 13} -> 0 attributed rows, groups kept: {'gender': 2, 'politics': 2, 'age': 2}


## Why AU: the same text split two ways

For one politics text, the groups of politics are unimodal and the groups of gender are not.

In [6]:
text = result.clean[result.clean["item_id"] == "t1"]            # a text polarized by politics
for dimension in ("politics", "gender"):
    r = result.tool.analyse_text(text["rating"], text[dimension])
    print(dimension, "->", r["reason"], r["group_dfu"])

politics -> AU {'left': 0.0, 'right': 0.0}
gender -> a group is not unimodal {'man': 0.75, 'woman': 0.5}


## Chance check

How many texts would a characteristic attribute if the ratings were dealt to the annotators of each text in every other possible way (group sizes fixed, the observed assignment left out)?

In [ ]:
from polaudit import compute_null

null = compute_null({"toy": result}, reps=200)
null["per_characteristic"].iloc[:, :9]